<a href="https://colab.research.google.com/github/srj1407/Practice/blob/main/Colab/From_APIs_To_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Todos

**🏋️ Todo 1: Context Management**
Our agent re-sends everything each iteration. Add summarization: after N iterations, compress the conversation history. Compare token usage before/after.

**🏋️ Todo 2: Planning Step**
Modify the agent to first create a PLAN (list of steps), then execute each one. Compare Plan→Execute vs pure ReAct on the same task.

**🏋️ Todo 3: Multi-Agent**
Create a 'researcher' (Wikipedia tools) and a 'coder' (file/code tools). Build an orchestrator that routes subtasks to the right agent.

**🏋️ Todo 4: Web Browsing**
Add a `fetch_webpage(url)` tool. The agent can now follow links. What new failure modes appear?



In [ ]:
!pip install openai requests -q
!pip install tavily-python

In [ ]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata
from tavily import TavilyClient

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["TAVILY"] = userdata.get("TAVILY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
tavily_client = TavilyClient(api_key=os.environ["TAVILY"])
TOOL_MODEL = "openrouter/hunter-alpha"

# TODO 1

In [ ]:
messages = [
        {"role": "system", "content":
            "You are a helpful research assistant."},
        {"role": "user", "content": "Hi"},
    ]
response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            temperature=0, max_tokens=800,
        )
print(response)

ChatCompletion(id='gen-1773856549-1BSTUmrUiBlhXfb9zBRY', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Hello!  😊 How can I assist you today? Whether you need help with research, writing, analysis, brainstorming, or just have a question—I'm here to help!  \n\nWhat can I do for you today?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='Okay, the user just said "Hi". That\'s a very minimal greeting. \n\nHmm, they might be testing if I\'m responsive, or perhaps they\'re just starting a conversation and haven\'t decided what to ask yet. Could also be a non-native English speaker keeping it simple. \n\nSince they didn\'t specify anything, I should keep it warm and open-ended. No need to overwhelm them with options right away - just show I\'m ready to help with whatever they need. \n\nThe smiley feels appropriate here to set a friendly tone. I\'ll list a few common categories

In [ ]:
NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web with Tavily for up-to-date information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
]

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

def tavily_search(**kwargs):
    # Pass ALL supported kwargs straight to Tavily
    results = tavily_client.search(**kwargs)
    return json.dumps(results)

NATIVE_FNS = {
    "tavily_search": tavily_search,
    "calculate": calculate_math,
    "get_current_date": get_current_date,
}

token_log = []


def run_native_agent(user_query, max_iterations=10, verbose=True):
    """
    Agent using native function calling.
    No regex — the API returns structured tool_calls.
    """
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools to find information. "
            "Think step by step. After gathering enough info, give a complete answer."},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} ---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        # Text response = done
        if finish == "stop" and msg.content:
            if verbose:
                print(f"✅ FINAL ANSWER:\n  {msg.content[:500]}")
            return {"answer": msg.content, "iterations": i + 1, "token_log": token_log}

        # Tool calls
        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)[:80]})")

                if fn_name in NATIVE_FNS:
                    result = NATIVE_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result[:150]}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break

    return {"answer": "Max iterations reached", "iterations": max_iterations, "token_log": token_log}

In [ ]:
result = run_native_agent(
    "What's the result when 21 is added to the result of 100 multiplied by the days returned from the difference in the birth dates of Prime Minister of India and LoP of India. "
)


🧑 USER: What's the result when 21 is added to the result of 100 multiplied by the days returned from the difference in the birth dates of Prime Minister of India and LoP of India. 

--- Iteration 1 ---
🔧 TOOL: tavily_search({"query": "current Prime Minister of India birth date"})
👁️  RESULT: {"query": "current Prime Minister of India birth date", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.in
🔧 TOOL: tavily_search({"query": "current Leader of Opposition LoP India birth date"})
👁️  RESULT: {"query": "current Leader of Opposition LoP India birth date", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https:/

--- Iteration 2 ---
🔧 TOOL: calculate({"expression": "(19 * 365 + 5) + (13 + 31 + 30 + 31 + 31 + 28 + 31 + 30 + 31 + 1)
👁️  RESULT: {"expression": "(19 * 365 + 5) + (13 + 31 + 30 + 31 + 31 + 28 + 31 + 30 + 31 + 19)", "result": 7215}

--- Iteration 3 ---
🔧 TOOL: calculate({"expression": "(100 * 7215) + 21

In [ ]:
print("\n" + "=" * 60)
print("TOKEN USAGE PER ITERATION")
print("=" * 60)
total = 0
for t in token_log:
    subtotal = t['in'] + t['out']
    total += subtotal
    bar = "█" * (t['in'] // 50)
    print(f"  Iter {t['iter']}: {t['in']:>5} in + {t['out']:>4} out = {subtotal:>5} │ {bar}")

print(f"\n  TOTAL: {total:,} tokens")
print(f"  → prompt_tokens GROWS each iteration — the context engineering problem.")


TOKEN USAGE PER ITERATION
  Iter 1:   478 in +  196 out =   674 │ █████████
  Iter 2:  1959 in +  720 out =  2679 │ ███████████████████████████████████████
  Iter 3:  2755 in +  124 out =  2879 │ ███████████████████████████████████████████████████████
  Iter 4:  2920 in +  200 out =  3120 │ ██████████████████████████████████████████████████████████

  TOTAL: 9,352 tokens
  → prompt_tokens GROWS each iteration — the context engineering problem.


In [ ]:
NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web with Tavily for up-to-date information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
]

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

def tavily_search(**kwargs):
    # Pass ALL supported kwargs straight to Tavily
    results = tavily_client.search(**kwargs)
    return json.dumps(results)

NATIVE_FNS = {
    "tavily_search": tavily_search,
    "calculate": calculate_math,
    "get_current_date": get_current_date,
}

token_log = []


def run_native_agent(user_query, max_iterations=15, verbose=True):
    """
    Agent using native function calling.
    No regex — the API returns structured tool_calls.
    """
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools to find information. "
            "Think step by step. After gathering enough info, give a complete answer."},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} ---")

        if i!=0 and i%3==0:
          summarizer_message = [
              {"role": "system", "content":
                  "You are a helpful research assistant. You are given a list of messages of conversation between AI and user. "
                  "You have to summarize and compress all the messages and return a single message containing all the relevant information till now."
                  "The message returned should be to the point and accurate so that this can be passed to the llm for taking the next steps for solving the user query."},
              {"role": "user", "content": f"""Return the summary for the following conversation:
              {messages}"""}
          ]
          if verbose:
            print(f"\n--- Summarising messages till now.. ---")
          response = client.chat.completions.create(
            model=TOOL_MODEL, messages=summarizer_message,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
          )
          print(response.choices[0].message.content)

          tokens_in = response.usage.prompt_tokens
          tokens_out = response.usage.completion_tokens
          token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

          messages = [
            {"role": "system", "content":
                f"""You are a helpful research assistant. You use tools to find information.
                You think step by step. After gathering enough info, you give a complete answer.
                You are being given a summary of conversation till now.
                You take the given info and proceed with the next steps.

                Conversation Summary:
                {messages}

                """},
            {"role": "user", "content": user_query},
         ]

          continue

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
        )

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        # Text response = done
        if finish == "stop" and msg.content:
            if verbose:
                print(f"✅ FINAL ANSWER:\n  {msg.content[:500]}")
            return {"answer": msg.content, "iterations": i + 1, "token_log": token_log}

        # Tool calls
        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)[:80]})")

                if fn_name in NATIVE_FNS:
                    result = NATIVE_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result[:150]}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break

    return {"answer": "Max iterations reached", "iterations": max_iterations, "token_log": token_log}

In [ ]:
result = run_native_agent(
    "What's the result when 21 is added to the result of 100 multiplied by the days returned from the difference in the birth dates of Prime Minister of India and LoP of India. "
)


🧑 USER: What's the result when 21 is added to the result of 100 multiplied by the days returned from the difference in the birth dates of Prime Minister of India and LoP of India. 

--- Iteration 1 ---
🔧 TOOL: tavily_search({"query": "Prime Minister of India birth date 2024"})
👁️  RESULT: {"query": "Prime Minister of India birth date 2024", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.mid-d
🔧 TOOL: tavily_search({"query": "Leader of Opposition LoP India 2024 birth date"})
👁️  RESULT: {"query": "Leader of Opposition LoP India 2024 birth date", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://ww

--- Iteration 2 ---
🔧 TOOL: calculate({"expression": "(19 * 365 + 5) + (13 + 31 + 30 + 31 + 31 + 28 + 31 + 30 + 31 + 1)
👁️  RESULT: {"expression": "(19 * 365 + 5) + (13 + 31 + 30 + 31 + 31 + 28 + 31 + 30 + 31 + 19)", "result": 7215}

--- Iteration 3 ---
🔧 TOOL: calculate({"expression": "100 * 7215 + 21"})
👁️  

In [ ]:
print("\n" + "=" * 60)
print("TOKEN USAGE PER ITERATION")
print("=" * 60)
total = 0
for t in token_log:
    subtotal = t['in'] + t['out']
    total += subtotal
    bar = "█" * (t['in'] // 50)
    print(f"  Iter {t['iter']}: {t['in']:>5} in + {t['out']:>4} out = {subtotal:>5} │ {bar}")

print(f"\n  TOTAL: {total:,} tokens")
print(f"  → prompt_tokens GROWS each iteration — the context engineering problem.")


TOKEN USAGE PER ITERATION
  Iter 1:   478 in +  203 out =   681 │ █████████
  Iter 2:  2606 in + 1116 out =  3722 │ ████████████████████████████████████████████████████
  Iter 3:  3798 in +  136 out =  3934 │ ███████████████████████████████████████████████████████████████████████████
  Iter 4:  5796 in +  374 out =  6170 │ ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Iter 5:  5822 in +  343 out =  6165 │ ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

  TOTAL: 20,672 tokens
  → prompt_tokens GROWS each iteration — the context engineering problem.
